# Title and Author Cleaning

Clean `title` and `author` fields from the Goodreads raw data while preserving the original values.

Outputs created:
- `title_clean`
- `title_key`
- `title_has_subtitle`
- `title_has_bracket_note`
- `author_clean`
- `primary_author`
- `author_key`
- `author_contributors_count`
- `author_had_role_labels`


In [ ]:
import re
import unicodedata
from pathlib import Path

import pandas as pd

RAW_DATA_PATH = Path('../data/raw/books_goodreads.csv')
if not RAW_DATA_PATH.exists():
    RAW_DATA_PATH = Path('data/raw/books_goodreads.csv')

df = pd.read_csv(RAW_DATA_PATH, dtype='string', keep_default_na=False)
df.shape


## Cleaning Decisions

- Keep raw `title` and `author` unchanged.
- Create display-safe cleaned fields by normalizing whitespace and removing control characters.
- Do not remove subtitles from `title_clean`; subtitles can be meaningful for nonfiction and series collections.
- Create `title_key` and `author_key` for matching/deduplication by lowercasing, removing accents, and reducing punctuation.
- Strip author role labels such as `(Goodreads Author)`, `(Illustrator)`, `(Translator)`, `(Editor)`, and `(Introduction)` from cleaned author names.
- Keep only the first listed contributor as `primary_author`; keep `author_clean` as all listed contributors after role-label cleanup.
- Flag rows where title or author cleaning changed the original values.

In [ ]:
CONTROL_CHARS_RE = re.compile(r'[\x00-\x1f\x7f]')
WHITESPACE_RE = re.compile(r'\s+')
ROLE_LABEL_RE = re.compile(r'\s*\([^)]*\)')

def normalize_whitespace(value):
    value = str(value or '')
    value = CONTROL_CHARS_RE.sub(' ', value)
    value = WHITESPACE_RE.sub(' ', value)
    return value.strip()

def strip_accents(value):
    normalized = unicodedata.normalize('NFKD', value)
    return ''.join(char for char in normalized if not unicodedata.combining(char))

def make_text_key(value):
    value = normalize_whitespace(value).casefold()
    value = strip_accents(value)
    value = value.replace('&', ' and ')
    value = re.sub(r'[^a-z0-9]+', ' ', value)
    return normalize_whitespace(value)

def split_author_contributors(author_value):
    # Split on commas only when they are outside parenthetical role labels.
    contributors = []
    current = []
    depth = 0
    for char in str(author_value or ''):
        if char == '(':
            depth += 1
        elif char == ')' and depth > 0:
            depth -= 1
        if char == ',' and depth == 0:
            contributors.append(''.join(current))
            current = []
        else:
            current.append(char)
    contributors.append(''.join(current))
    return [normalize_whitespace(part) for part in contributors if normalize_whitespace(part)]

def clean_author_name(value):
    value = normalize_whitespace(value)
    value = ROLE_LABEL_RE.sub('', value)
    return normalize_whitespace(value)


## Clean Titles

In [ ]:
df_titles_authors = df.copy()

df_titles_authors['title_clean'] = df_titles_authors['title'].map(normalize_whitespace).astype('string')
df_titles_authors['title_key'] = df_titles_authors['title_clean'].map(make_text_key).astype('string')
df_titles_authors['title_has_subtitle'] = df_titles_authors['title_clean'].str.contains(':', regex=False)
df_titles_authors['title_has_bracket_note'] = df_titles_authors['title_clean'].str.contains(r'\[[^\]]+\]|\([^\)]+\)', regex=True)
df_titles_authors['title_clean_changed'] = df_titles_authors['title_clean'].ne(df_titles_authors['title'])

df_titles_authors[['title', 'title_clean', 'title_key', 'title_has_subtitle', 'title_has_bracket_note']].head(20)


## Clean Authors

In [ ]:
author_contributors = df_titles_authors['author'].map(split_author_contributors)
cleaned_contributors = author_contributors.map(lambda parts: [clean_author_name(part) for part in parts if clean_author_name(part)])

df_titles_authors['author_clean'] = cleaned_contributors.map(lambda parts: ', '.join(parts)).astype('string')
df_titles_authors['primary_author'] = cleaned_contributors.map(lambda parts: parts[0] if parts else pd.NA).astype('string')
df_titles_authors['author_key'] = df_titles_authors['primary_author'].fillna('').map(make_text_key).astype('string')
df_titles_authors['author_contributors_count'] = cleaned_contributors.map(len).astype('Int64')
df_titles_authors['author_had_role_labels'] = df_titles_authors['author'].str.contains(r'\([^)]*\)', regex=True)
df_titles_authors['author_clean_changed'] = df_titles_authors['author_clean'].ne(df_titles_authors['author'])

df_titles_authors[['author', 'author_clean', 'primary_author', 'author_key', 'author_contributors_count', 'author_had_role_labels']].head(20)


## Cleaning Audit

In [ ]:
title_author_cleaning_summary = pd.DataFrame({
    'metric': [
        'total_rows',
        'blank_title_clean',
        'blank_primary_author',
        'title_clean_changed',
        'author_clean_changed',
        'title_has_subtitle',
        'title_has_bracket_note',
        'author_had_role_labels',
        'multiple_cleaned_contributors',
        'duplicate_title_key_rows',
        'duplicate_title_author_key_rows',
    ],
    'rows': [
        len(df_titles_authors),
        int(df_titles_authors['title_clean'].str.strip().eq('').sum()),
        int(df_titles_authors['primary_author'].fillna('').str.strip().eq('').sum()),
        int(df_titles_authors['title_clean_changed'].sum()),
        int(df_titles_authors['author_clean_changed'].sum()),
        int(df_titles_authors['title_has_subtitle'].sum()),
        int(df_titles_authors['title_has_bracket_note'].sum()),
        int(df_titles_authors['author_had_role_labels'].sum()),
        int(df_titles_authors['author_contributors_count'].gt(1).sum()),
        int(df_titles_authors['title_key'].duplicated(keep=False).sum()),
        int(df_titles_authors[['title_key', 'author_key']].duplicated(keep=False).sum()),
    ],
})

title_author_cleaning_summary


## Spot Check: Rows Changed by Author Cleaning

In [ ]:
df_titles_authors.loc[
    df_titles_authors['author_clean_changed'],
    ['title_clean', 'author', 'author_clean', 'primary_author', 'author_contributors_count']
].head(30)


## Spot Check: Potential Duplicate Works After Cleaning

In [ ]:
duplicate_work_keys = df_titles_authors[['title_key', 'author_key']].duplicated(keep=False)

df_titles_authors.loc[
    duplicate_work_keys,
    ['title', 'title_clean', 'author', 'primary_author', 'title_key', 'author_key']
].sort_values(['title_key', 'author_key']).head(40)


## Final Output Columns

Carry these columns forward into the cleaned working dataset. Keep raw `title` and `author` too, so display labels can be audited later.

In [ ]:
title_author_output_columns = [
    'bookId',
    'title',
    'title_clean',
    'title_key',
    'title_has_subtitle',
    'title_has_bracket_note',
    'author',
    'author_clean',
    'primary_author',
    'author_key',
    'author_contributors_count',
    'author_had_role_labels',
]

df_titles_authors[title_author_output_columns].head(20)
